In [1]:
#Sheet Pvt

In [2]:
import pandas as pd

# ==============================
# LOAD DATA
# ==============================
df = pd.read_excel("PTC draft.xlsx")

# Create Year-Month key
df["Year_Month"] = (
    df["Year Reqrd."].astype(str) + "-" +
    df["Mon Reqrd."].astype(str).str.zfill(2)
)

# ==============================
# PIVOT Qty Open BY MONTH
# ==============================
pivot_df = (
    df.pivot_table(
        index=["PN AL78", "PN used", "Description", "Seqnc"],
        columns="Year_Month",
        values="Qty Open",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

# ==============================
# AGGREGATE Qty Resvered FROM ORIGINAL DF
# ==============================
# ⚠️ Use SUM or MAX depending on ERP behavior
reserved_df = (
    df.groupby(["PN AL78", "PN used", "Description", "Seqnc"])["Qty Resvered"]
    .sum()      # change to .max() if duplicated per row
    .reset_index()
)

# ==============================
# MERGE Qty Resvered INTO PIVOT
# ==============================
pivot_df = pivot_df.merge(
    reserved_df,
    on=["PN AL78", "PN used", "Description", "Seqnc"],
    how="left"
)

# ==============================
# MOVE Qty Resvered AFTER Seqnc
# ==============================
cols = pivot_df.columns.tolist()
cols.remove("Qty Resvered")

seq_idx = cols.index("Seqnc")
cols.insert(seq_idx + 1, "Qty Resvered")

pivot_df = pivot_df[cols]


In [3]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [4]:
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str)

pivot_df = (
    pivot_df
    .sort_values(by=["PN AL78", "Seqnc"], ascending=[True, True])
    .reset_index(drop=True)
)


In [5]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [6]:
pivot_df.insert(
    loc=0,
    column="PN",
    value=pivot_df["PN AL78"]
)

# === CONFIG ===
current_period = pd.Period("2026-03", freq="M")

# === PREP RAW DATA ===
df["Period"] = pd.PeriodIndex(
    df["Year Reqrd."].astype(str) + "-" +
    df["Mon Reqrd."].astype(str).str.zfill(2),
    freq="M"
)

# === FILTER 7 MONTHS PRIOR ===
prior_7m_df = df[
    (df["Period"] < current_period) &
    (df["Period"] >= current_period - 7)
]

# === ROW-LEVEL AGGREGATION (PN + Seqnc) ===
prior_7m_sum = (
    prior_7m_df
    .groupby(["PN AL78", "Seqnc"], as_index=False)["Qty Open"]
    .sum()
    .rename(columns={"Qty Open": "7 Months Prior"})
)

# === MERGE INTO PIVOT TABLE ===
pivot_df = pivot_df.merge(
    prior_7m_sum,
    on=["PN AL78", "Seqnc"],
    how="left"
)

pivot_df["7 Months Prior"] = pivot_df["7 Months Prior"].fillna(0).astype(int)


In [7]:
print(pivot_df)

                PN       PN AL78      PN used             Description  Seqnc  \
0           101843        101843    010184300                    SHIM      1   
1           102020        102020    010202000  SCREW,HEXAGON HEAD CAP      1   
2           103023        103023    010302300  SCREW,HEXAGON HEAD CAP      1   
3           106069        106069    010606900  SCREW,HEXAGON HEAD CAP      1   
4           106069        106069    010606900  SCREW,HEXAGON HEAD CAP      2   
...            ...           ...          ...                     ...    ...   
2322  S   962    E  S   962    E  S00096200 E               DRAINCOCK      2   
2323  S  1040    A  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1   
2324  S  1040    A  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2   
2325       S  2268       S  2268    S00226800          FITTING,GREASE      1   
2326       S 16240       S 16240    S01624000          RING,RETAINING      1   

      Qty Resvered  2025-08  2025-09  2

In [8]:
# =========================
# OPEN ORDERS — FINAL FIX
# =========================
month_cols = [
    c for c in pivot_df.columns
    if isinstance(c, str)
    and c[:4].isdigit()
    and "-" in c
]

In [9]:
# Take current column order
cols = pivot_df.columns.tolist()

# Remove PN from its current position
cols.remove("PN")

# Insert PN right after "03" (2026-03)
last_month = max(month_cols)
insert_pos = cols.index(last_month) + 1

cols.insert(insert_pos, "PN")

# Reorder dataframe
pivot_df = pivot_df[cols]


In [10]:
def get_month(col_list, idx):
    return col_list[idx] if idx < len(col_list) else None


In [11]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [12]:
# =========================
# OPEN ORDERS — FINAL FIX (SAFE)
# =========================

month_cols = [
    c for c in pivot_df.columns
    if isinstance(c, str)
    and c[:4].isdigit()
    and "-" in c
]

# Ensure numeric
pivot_df[month_cols] = (
    pivot_df[month_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# Base month = Mar 2026
base_month = "2026-03"

if base_month not in month_cols:
    raise ValueError(f"{base_month} not found in month columns")

base_idx = month_cols.index(base_month)

# Safe month references
m0 = get_month(month_cols, base_idx)
m1 = get_month(month_cols, base_idx + 1)
m2 = get_month(month_cols, base_idx + 2)
m3 = get_month(month_cols, base_idx + 3)

pivot_df["OO CM"]  = pivot_df[m0] + (pivot_df[m1] if m1 else 0)
pivot_df["OO NM"]  = pivot_df[m2] if m2 else 0
pivot_df["OO N2M"] = pivot_df[m3] if m3 else 0
pivot_df["OO N3M"] = 0

# Ensure int
pivot_df[["OO CM","OO NM","OO N2M","OO N3M"]] = (
    pivot_df[["OO CM","OO NM","OO N2M","OO N3M"]].astype(int)
)


In [13]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [14]:
import pandas as pd
import numpy as np

# === READ FILES ===
pncheck_df = pd.read_excel("Pvt chkPN draft.xlsx", dtype=str)
FCPTC_df = pd.read_excel(
    "PTC OvH Forecast Mar2026.xlsx",
    sheet_name="Pvt",
    skiprows=2
)

# === NORMALIZE PN FORMAT ===
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN AL78"] = pncheck_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN Final"] = pncheck_df["PN Final"].astype(str).str.strip()

# =========================================================
# 1️⃣ ADD PN.Final (lookup PN AL78 → PN Final)
# =========================================================
pivot_df = pivot_df.merge(
    pncheck_df[["PN AL78", "PN Final"]]
        .rename(columns={"PN Final": "PN.Final"}),
    on="PN AL78",
    how="left"
)

# === PLACE 'PN.Final' AFTER 'OO N3M' ===
cols = pivot_df.columns.tolist()
cols.remove("PN.Final")
cols.insert(cols.index("OO N3M") + 1, "PN.Final")
pivot_df = pivot_df[cols]

# =========================================================
# 2️⃣ FC LOOKUP USING *PN.Final*
# =========================================================
fc_lookup = (
    FCPTC_df[["PN Final","M1 Final", "M2 Final", "M3 Final", "M4 Final", "M5 Final"]]
    .rename(columns={
        "PN Final": "PN.Final",
        "M1 Final": "FC CM",
        "M2 Final": "FC NM",
        "M3 Final": "FC N2M",
        "M4 Final": "FC N3M",
        "M5 Final": "FC N4M"
    })
)

pivot_df = pivot_df.merge(
    fc_lookup,
    on="PN.Final",
    how="left"
)

# === CLEAN FC VALUES (#N/A → 0, remove .0) ===
for col in ["FC CM", "FC NM", "FC N2M", "FC N3M", "FC N4M"]:
    pivot_df[col] = (
        pd.to_numeric(pivot_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

# === PLACE FC COLUMNS AFTER 'PN.Final' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("PN.Final") + 1

for col in ["FC CM", "FC NM", "FC N2M", "FC N3M", "FC N4M"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [15]:
print(FCPTC_df.columns)

Index(['AGC', 'PART NO', 'M1.', 'M2.', 'M3.', 'M4.', 'M5.', 'Unnamed: 7',
       'PN no suffx', 'Rplcmt', 'NPN1', 'NPN2', 'NPN3', 'NPN4', 'NPN5',
       'PN.Final', 'M1', 'M2', 'M3', 'M4', 'M5', 'Unnamed: 21', 'PN Final',
       'M1 Final', 'M2 Final', 'M3 Final', 'M4 Final', 'M5 Final'],
      dtype='object')


In [16]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [17]:
# Check duplicates in pncheck_df
pncheck_df["PN AL78"].value_counts().loc[lambda x: x > 1]

# Check duplicates in FCPTC_df
FCPTC_df["PN Final"].value_counts().loc[lambda x: x > 1]

Series([], Name: count, dtype: int64)

In [18]:
# === ADD 2 EMPTY COLUMNS ===
pivot_df[""] = ""
pivot_df["  "] = ""

# === ADD LONG PN ===
pivot_df["Long PN"] = pivot_df["PN used"]

# === MOVE COLUMNS TO THE RIGHT OF 'FC N4M' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("FC N4M") + 1

for col in ["", "  ", "Long PN"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [19]:
import numpy as np

# === ENSURE NUMERIC (SAFETY) ===
num_cols = [
    "FC CM", "OO CM",
    "FC NM", "OO NM",
    "FC N2M", "OO N2M",
    "FC N3M", "OO N3M"
]

for col in num_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === CALCULATIONS (ROW-LEVEL, EXCEL-EQUIVALENT) ===
pivot_df["Curr. Month"] = (
    pivot_df[["FC CM", "OO CM"]].max(axis=1) +
    pivot_df[["FC NM", "OO NM"]].max(axis=1)
)

pivot_df["Req CM"] = pivot_df[["FC N2M", "OO N2M"]].max(axis=1)

pivot_df["Req NM"] = pivot_df[["FC N3M", "OO N3M"]].max(axis=1)

# === BLANK COLUMNS ===
pivot_df["Req N2M"] = ""
pivot_df["Req N3M"] = ""

# === PLACE COLUMNS AFTER 'Long PN' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("Long PN") + 1

for col in ["Curr. Month", "Req CM", "Req NM", "Req N2M", "Req N3M"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [20]:
# === READ PARTS MOVEMENT DATA ===
partsmovementPTC_df = pd.read_excel(
    "pmovdcE PTC DR 28Mar2026.xlsx",
    sheet_name="OH All"
)

# === NORMALIZE PN KEYS (CRITICAL) ===
partsmovementPTC_df["PN Final"] = (
    partsmovementPTC_df["PN Final"]
    .astype(str)
    .str.strip()
    .str.upper()
)

pivot_df["PN.Final"] = (
    pivot_df["PN.Final"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# === AGGREGATE OH PER Last PN.1 ===
oh_lookup = (
    partsmovementPTC_df
    .groupby("PN Final", as_index=False)["OH All"]
    .sum()
    .rename(columns={
        "PN Final": "PN.Final",
        "OH All": "OH"
    })
)

# === MERGE USING PN.Final ===
pivot_df = pivot_df.merge(
    oh_lookup,
    on="PN.Final",
    how="left"
)

# === ENSURE NUMERIC (KEEP NaN) ===
pivot_df["OH"] = pd.to_numeric(
    pivot_df["OH"],
    errors="coerce"
)

# === PLACE COLUMN AFTER 'Req N3M' ===
cols = pivot_df.columns.tolist()
cols.remove("OH")
cols.insert(cols.index("Req N3M") + 1, "OH")

pivot_df = pivot_df[cols]


In [21]:
print(partsmovementPTC_df.columns)

Index(['Brc', 'Agc', 'PN', 'P/N', 'Desc', 'DN Price', 'DR/\nNDR', 'OH', 'OO',
       'Book', 'Alloc\nIn', 'Alloc\nOut', 'OH\nAll', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'PN.1',
       'OHAll', 'Unnamed: 20', 'PN-NPN\n(replcmt)', 'NPN\n(replcmt)', 'NPN1',
       'NPN2', 'NPN3', 'NPN4', 'NPN5', 'PN Final', 'OHAll.1', 'Unnamed: 30',
       'Unnamed: 31', 'Unnamed: 32', 'Last PN', 'OH All'],
      dtype='object')


In [22]:
# =========================
# RECEIVING AND TRANSIT
# =========================

# === READ DATA ===
osgrr_df = pd.read_excel(
    "Outstd.GRR PTC 30Mar2026.xlsx",
    sheet_name="Pvt",
    skiprows=3
)

shipment_df = pd.read_excel(
    "PTA PTC Weekly Shipment Invoice 28Mar2026.xlsx",
    sheet_name="noGRR",
    skiprows=2
)

# === AGGREGATE SI (by Last PN) ===
rcv_si = (
    osgrr_df
    .groupby("Last PN", as_index=False)["InRcv.1"]
    .sum()
    .rename(columns={"Last PN": "PN.Final", "InRcv.1": "In Rcv SI"})
)

tr_si = (
    osgrr_df
    .groupby("Last PN", as_index=False)["InTr.1"]
    .sum()
    .rename(columns={"Last PN": "PN.Final", "InTr.1": "In Tr SI"})
)

# === AGGREGATE NON-SI (by PN Final) ===
rcv = (
    shipment_df
    .groupby("PN Final", as_index=False)["InRcv.1"]
    .sum()
    .rename(columns={"PN Final": "PN.Final", "InRcv.1": "In Rcv"})
)

tr = (
    shipment_df
    .groupby("PN Final", as_index=False)["InTr.1"]
    .sum()
    .rename(columns={"PN Final": "PN.Final", "InTr.1": "In Tr"})
)

# === MERGE ALL LOOKUPS USING PN.Final ===
for df_add in [rcv_si, tr_si, rcv, tr]:
    pivot_df = pivot_df.merge(
        df_add,
        on="PN.Final",
        how="left"
    )

# === CLEAN NaN → 0 ===
for col in ["In Rcv SI", "In Tr SI", "In Rcv", "In Tr"]:
    pivot_df[col] = (
        pd.to_numeric(pivot_df[col], errors="coerce")
        .fillna(0)
        .astype(int)
    )

# === TOTAL COLUMNS ===
pivot_df["In Rcv Total"] = pivot_df["In Rcv SI"] + pivot_df["In Rcv"]
pivot_df["In Tr Total"]  = pivot_df["In Tr SI"]  + pivot_df["In Tr"]

# === PLACE COLUMNS BESIDE 'OH' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("OH") + 1

for col in [
    "In Rcv Total", "In Tr Total",
    "In Rcv SI", "In Tr SI",
    "In Rcv", "In Tr"
]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [23]:
import numpy as np

# === ENSURE NUMERIC INPUTS ===
check_cols = [
    "OH",
    "Curr. Month",
    "In Rcv Total",
    "In Tr Total",
    "7 Months Prior",
    "OO CM",
    "OO NM"
]

for col in check_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === chk OH to CM LOGIC (EXCEL-EQUIVALENT) ===
pivot_df["chk OH to CM"] = np.where(
    pivot_df["OH"] < pivot_df["Curr. Month"],
    np.where(
        pivot_df["Curr. Month"] <= (
            pivot_df["OH"]
            + pivot_df["In Rcv Total"]
            + pivot_df["In Tr Total"]
            + pivot_df["7 Months Prior"]
            + pivot_df["OO CM"]
            + pivot_df["OO NM"]
        ),
        "OK",
        "NG"
    ),
    "OK"
)

# === PLACE COLUMN AFTER 'In Tr Total' ===
cols = pivot_df.columns.tolist()
cols.remove("chk OH to CM")
cols.insert(cols.index("In Tr") + 1, "chk OH to CM")

pivot_df = pivot_df[cols]


In [24]:
# === ENSURE NUMERIC INPUTS ===
nm_cols = [
    "OH",
    "In Rcv Total",
    "In Tr Total",
    "Curr. Month",
    "Req CM"
]

for col in nm_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === chk OH to NM (EXCEL-EQUIVALENT) ===
pivot_df["chk OH to NM"] = (
    pivot_df["OH"]
    + pivot_df["In Rcv Total"]
    + pivot_df["In Tr Total"]
    - pivot_df["Curr. Month"]
    - pivot_df["Req CM"]
)

# === PLACE COLUMN AFTER 'chk OH to CM' ===
cols = pivot_df.columns.tolist()
cols.remove("chk OH to NM")
cols.insert(cols.index("chk OH to CM") + 1, "chk OH to NM")

pivot_df = pivot_df[cols]


In [25]:
# === ENSURE NUMERIC INPUTS ===
req_cols = ["Curr. Month", "Req CM"]

for col in req_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === Req CM Total (EXCEL-EQUIVALENT) ===
pivot_df["Req CM Total"] = (
    pivot_df["Curr. Month"] + pivot_df["Req CM"]
)

# === PLACE COLUMN AFTER 'chk OH to NM' ===
cols = pivot_df.columns.tolist()
cols.remove("Req CM Total")
cols.insert(cols.index("chk OH to NM") + 1, "Req CM Total")

pivot_df = pivot_df[cols]


In [26]:
# === ENSURE NUMERIC INPUTS ===
opor_cols = ["7 Months Prior", "OO CM"]

for col in opor_cols:
    pivot_df[col] = pd.to_numeric(pivot_df[col], errors="coerce").fillna(0)

# === OpOrd CM (EXCEL-EQUIVALENT) ===
pivot_df["OpOrd CM"] = (
    pivot_df["7 Months Prior"] + pivot_df["OO CM"]
)

# === PLACE COLUMN AFTER 'Req CM Total' ===
cols = pivot_df.columns.tolist()
cols.remove("OpOrd CM")
cols.insert(cols.index("Req CM Total") + 1, "OpOrd CM")

pivot_df = pivot_df[cols]


In [27]:
# === ADD DESCRIPTION 2 ===
pivot_df["Description 2"] = pivot_df["Description"]

# === PLACE COLUMN AFTER 'OpORd CM' ===
cols = pivot_df.columns.tolist()
cols.remove("Description 2")
cols.insert(cols.index("OpOrd CM") + 1, "Description 2")

pivot_df = pivot_df[cols]


In [28]:
# === READ SOURCE FILE ===
ptc_src_df = pd.read_excel("PTC draft.xlsx")

# === BUILD LOOKUP KEY IN SOURCE ===
ptc_src_df["lookup_key"] = (
    ptc_src_df["Seqnc"].astype(str) + "." +
    ptc_src_df["PN AL78"].astype(str)
)

esd_lookup = (
    ptc_src_df[["lookup_key", "ESD Mon"]]
    .drop_duplicates()
)

# === BUILD LOOKUP KEY IN PIVOT_DF ===
pivot_df["lookup_key"] = (
    pivot_df["Seqnc"].astype(str) + "." +
    pivot_df["PN AL78"].astype(str)
)

# === MERGE (VLOOKUP STYLE) ===
pivot_df = pivot_df.merge(
    esd_lookup,
    on="lookup_key",
    how="left"
)

# === CLEAN UP ===
pivot_df["Mon. ESD"] = pivot_df["ESD Mon"].fillna("")

pivot_df.drop(columns=["lookup_key", "ESD Mon"], inplace=True)

# === PLACE COLUMN AFTER 'Description 2' ===
cols = pivot_df.columns.tolist()
cols.remove("Mon. ESD")
cols.insert(cols.index("Description 2") + 1, "Mon. ESD")

pivot_df = pivot_df[cols]
# === CONVERT Mon. ESD TO INTEGER (NO .0, BLANK-SAFE) ===
pivot_df["Mon. ESD"] = (
    pd.to_numeric(pivot_df["Mon. ESD"], errors="coerce")
    .astype("Int64")   # nullable integer
)


In [29]:
pivot_df = pivot_df.rename(columns={
    "('Qty Resvered', '')": "Qty Resvered"
})


In [30]:
# === PLACE COLUMN AFTER 'Mon. ESD' ===
cols = pivot_df.columns.tolist()
#cols.remove("Qty Resvered")
cols.insert(cols.index("Mon. ESD") + 1, "Qty Resvered")

pivot_df = pivot_df[cols]


In [31]:
print(pivot_df.columns)

Index(['PN AL78', 'PN used', 'Description', 'Seqnc', 'Qty Resvered', '2025-08',
       '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02',
       '2026-03', '2026-04', '2026-05', '2026-06', 'PN', '7 Months Prior',
       'OO CM', 'OO NM', 'OO N2M', 'OO N3M', 'PN.Final', 'FC CM', 'FC NM',
       'FC N2M', 'FC N3M', 'FC N4M', '', '  ', 'Long PN', 'Curr. Month',
       'Req CM', 'Req NM', 'Req N2M', 'Req N3M', 'OH', 'In Rcv Total',
       'In Tr Total', 'In Rcv SI', 'In Tr SI', 'In Rcv', 'In Tr',
       'chk OH to CM', 'chk OH to NM', 'Req CM Total', 'OpOrd CM',
       'Description 2', 'Mon. ESD', 'Qty Resvered'],
      dtype='object')


In [32]:
print("Qty Resvered exists:", "Qty Resvered" in pivot_df.columns)
print(pivot_df["Qty Resvered"])


Qty Resvered exists: True
      Qty Resvered  Qty Resvered
0                0             0
1                0             0
2                0             0
3                0             0
4                0             0
...            ...           ...
2322             0             0
2323             0             0
2324             0             0
2325             2             2
2326             2             2

[2327 rows x 2 columns]


In [33]:
print(pivot_df)

           PN AL78      PN used             Description  Seqnc  Qty Resvered  \
0           101843    010184300                    SHIM      1             0   
1           102020    010202000  SCREW,HEXAGON HEAD CAP      1             0   
2           103023    010302300  SCREW,HEXAGON HEAD CAP      1             0   
3           106069    010606900  SCREW,HEXAGON HEAD CAP      1             0   
4           106069    010606900  SCREW,HEXAGON HEAD CAP      2             0   
...            ...          ...                     ...    ...           ...   
2322  S   962    E  S00096200 E               DRAINCOCK      2             0   
2323  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      1             0   
2324  S  1040    A  S00104000 A      ELBOW,MALE ADAPTER      2             0   
2325       S  2268    S00226800          FITTING,GREASE      1             2   
2326       S 16240    S01624000          RING,RETAINING      1             2   

      2025-08  2025-09  2025-10  2025-1

In [34]:
# === READ PN CHECK FILE ===
pncheck_df = pd.read_excel("Pvt chkPN draft.xlsx", dtype=str)

# === FORCE STRING (IMPORTANT) ===
pivot_df["PN AL78"] = pivot_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN AL78"] = pncheck_df["PN AL78"].astype(str).str.strip()
pncheck_df["PN Final"] = pncheck_df["PN Final"].astype(str).str.strip()

# === MERGE: PN AL78 → PN Final ===
pivot_df = pivot_df.merge(
    pncheck_df[["PN AL78", "PN Final"]].drop_duplicates(),
    on="PN AL78",
    how="left"
)



In [35]:
print(pivot_df.columns)

Index(['PN AL78', 'PN used', 'Description', 'Seqnc', 'Qty Resvered', '2025-08',
       '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02',
       '2026-03', '2026-04', '2026-05', '2026-06', 'PN', '7 Months Prior',
       'OO CM', 'OO NM', 'OO N2M', 'OO N3M', 'PN.Final', 'FC CM', 'FC NM',
       'FC N2M', 'FC N3M', 'FC N4M', '', '  ', 'Long PN', 'Curr. Month',
       'Req CM', 'Req NM', 'Req N2M', 'Req N3M', 'OH', 'In Rcv Total',
       'In Tr Total', 'In Rcv SI', 'In Tr SI', 'In Rcv', 'In Tr',
       'chk OH to CM', 'chk OH to NM', 'Req CM Total', 'OpOrd CM',
       'Description 2', 'Mon. ESD', 'Qty Resvered', 'PN Final'],
      dtype='object')


In [36]:
# === ADD EMPTY COLUMN AFTER 'PN Final' ===
pivot_df["   "] = ""

# === ADD chk PN COLUMN ===
pivot_df["chk PN"] = np.where(
    pivot_df["PN.Final"].astype(str) == pivot_df["PN AL78"].astype(str),
    "OK",
    "NG"
)

# === PLACE COLUMNS AFTER 'PN Final' ===
cols = pivot_df.columns.tolist()
insert_at = cols.index("PN.Final") + 1

for col in ["   ", "chk PN"]:
    cols.remove(col)
    cols.insert(insert_at, col)
    insert_at += 1

pivot_df = pivot_df[cols]


In [37]:
print(pivot_df.columns)

Index(['PN AL78', 'PN used', 'Description', 'Seqnc', 'Qty Resvered',
       'Qty Resvered', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12',
       '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', 'PN',
       '7 Months Prior', 'OO CM', 'OO NM', 'OO N2M', 'OO N3M', 'PN.Final',
       '   ', 'chk PN', 'FC CM', 'FC NM', 'FC N2M', 'FC N3M', 'FC N4M', '',
       '  ', 'Long PN', 'Curr. Month', 'Req CM', 'Req NM', 'Req N2M',
       'Req N3M', 'OH', 'In Rcv Total', 'In Tr Total', 'In Rcv SI', 'In Tr SI',
       'In Rcv', 'In Tr', 'chk OH to CM', 'chk OH to NM', 'Req CM Total',
       'OpOrd CM', 'Description 2', 'Mon. ESD', 'Qty Resvered', 'Qty Resvered',
       'PN Final'],
      dtype='object')


In [38]:
# Get columns
cols = pivot_df.columns.tolist()

# Find anchor indexes
seq_idx = cols.index("Seqnc")
esd_idx = cols.index("Mon. ESD")

# Find all Qty Resvered positions
qty_idx = [i for i, c in enumerate(cols) if c == "Qty Resvered"]

print("All Qty Resvered positions:", qty_idx)

# We want to keep:
keep_positions = [seq_idx + 1, esd_idx + 1]

# Rename the two we keep
for i in qty_idx:
    if i == seq_idx + 1:
        cols[i] = "Qty Resvered"
    elif i == esd_idx + 1:
        cols[i] = "Qty Resvered 2"
    else:
        cols[i] = f"DROP_QTY_{i}"   # mark others for deletion

# Apply renamed columns
pivot_df.columns = cols

# Drop unwanted Qty Resvered columns
pivot_df = pivot_df.drop(columns=[c for c in pivot_df.columns if c.startswith("DROP_QTY_")])


All Qty Resvered positions: [4, 5, 52, 53]


In [39]:
print(pivot_df.columns)

Index(['PN AL78', 'PN used', 'Description', 'Seqnc', 'Qty Resvered', '2025-08',
       '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02',
       '2026-03', '2026-04', '2026-05', '2026-06', 'PN', '7 Months Prior',
       'OO CM', 'OO NM', 'OO N2M', 'OO N3M', 'PN.Final', '   ', 'chk PN',
       'FC CM', 'FC NM', 'FC N2M', 'FC N3M', 'FC N4M', '', '  ', 'Long PN',
       'Curr. Month', 'Req CM', 'Req NM', 'Req N2M', 'Req N3M', 'OH',
       'In Rcv Total', 'In Tr Total', 'In Rcv SI', 'In Tr SI', 'In Rcv',
       'In Tr', 'chk OH to CM', 'chk OH to NM', 'Req CM Total', 'OpOrd CM',
       'Description 2', 'Mon. ESD', 'Qty Resvered 2', 'PN Final'],
      dtype='object')


In [40]:
pivot_df.to_excel("Pvt draft.xlsx", index=False)